In [17]:
import pandas as pd 
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 경로를 지정하여 Graphviz를 설치한 경우, 다음과 같이 경로를 설정할 수 있습니다.
from sklearn.tree import export_graphviz
import graphviz

In [18]:
# HPO를 위한 설정
from sklearn.model_selection import train_test_split, GridSearchCV

In [19]:
# 데이터 로드
wine = load_wine()

In [20]:
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

In [21]:
# 모형 학습
# 특성과 타겟의 데이터를 분리
X = df.drop('target', axis=1)
y = df['target']    

In [22]:
# 학습 데이터와 테스트 데이터로 분리(80% 학습, 20% 테스트)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
DecisionTreeClassifier?

Init signature: DecisionTreeClassifier(*args, **kwargs)
Docstring:     
A decision tree classifier.

Read more in the :ref:`User Guide <tree>`.

Parameters
----------
criterion : {"gini", "entropy", "log_loss"}, default="gini"
    The function to measure the quality of a split. Supported criteria are
    "gini" for the Gini impurity and "log_loss" and "entropy" both for the
    Shannon information gain, see :ref:`tree_mathematical_formulation`.

splitter : {"best", "random"}, default="best"
    The strategy used to choose the split at each node. Supported
    strategies are "best" to choose the best split and "random" to choose
    the best random split.

max_depth : int, default=None
    The maximum depth of the tree. If None, then nodes are expanded until
    all leaves are pure or until all leaves contain less than
    min_samples_split samples.

min_samples_split : int or float, default=2
    The minimum number of samples required to split an internal node:

    - If int, then cons

In [30]:
# Hyperparameter Optimization (HPO) 설정 '수기' 변경
clf_mannual = DecisionTreeClassifier(criterion='gini',                                      
                                     max_depth=10,
                                     min_samples_split=5,
                                     min_samples_leaf=5,
                                     splitter='random', 
                                     random_state=4, 
                                     )
clf_mannual.fit(X_train, y_train)
y_test_pred_mannual = clf_mannual.predict(X_test)
accuracy_mannual = accuracy_score(y_test, y_test_pred_mannual)
print(f"수기 설정 정확도: {accuracy_mannual:.4f}")

수기 설정 정확도: 0.8056


In [32]:
# Hyperparameter Tuning (HPO) 설정 '자동' 변경
# GridSearchCV를 사용하여 최적의 하이퍼파라미터 탐색

param_grid = {
    "criterion" : ['gini', 'entropy'],
    "max_depth" : [2,3,4,5],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4],
}

In [33]:
# HPO 및 Fitting
clf_grid = DecisionTreeClassifier(random_state=42)

# core
grid_search = GridSearchCV(estimator=clf_grid, param_grid=param_grid, cv=5)
# Fitting
grid_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터와 정확도 출력
print(f"최적의 하이퍼파라미터: {grid_search.best_params_}")
print(f"최적의 정확도: {grid_search.best_score_}")

최적의 하이퍼파라미터: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
최적의 정확도: 0.9224137931034484


In [34]:
best_model = grid_search.best_estimator_

y_pred_grid = best_model.predict(X_test)
accuracy_grid = accuracy_score(y_test, y_pred_grid)

In [35]:
# 최적의 모델로 테스트 데이터 예측 및 평가
print(f"GridSearchCV 설정 정확도: {accuracy_grid:.4f}")

GridSearchCV 설정 정확도: 0.9444


In [37]:
grid_search.cv_results_

{'mean_fit_time': array([0.00354161, 0.00315351, 0.0034647 , 0.00322685, 0.00302529,
        0.00288601, 0.00284872, 0.00386834, 0.0032227 , 0.00349808,
        0.00331511, 0.00296164, 0.00324926, 0.0032548 , 0.00302005,
        0.00297852, 0.00295105, 0.00292783, 0.00319314, 0.00306997,
        0.00312166, 0.00386806, 0.00535893, 0.00505681, 0.00498037,
        0.00586586, 0.00510049, 0.00446339, 0.00428338, 0.00433335,
        0.00468493, 0.00468268, 0.0042016 , 0.00422158, 0.00403814,
        0.00406394, 0.0043982 , 0.00538769, 0.00412574, 0.00454745,
        0.00433869, 0.00407887, 0.00414734, 0.00428944, 0.00431967,
        0.0052844 , 0.00471182, 0.0046638 , 0.00466914, 0.00453115,
        0.00462828, 0.00442505, 0.00450473, 0.00497003, 0.00486493,
        0.0048945 , 0.00460196, 0.00463948, 0.00515265, 0.00518899,
        0.00492802, 0.00488572, 0.0045331 , 0.00520864, 0.0047061 ,
        0.0057457 , 0.00462265, 0.00519156, 0.00452676, 0.00486741,
        0.00493946, 0.00447187]